# SciGRID Network

*Adapted from: [PyPSA Examples – SciGRID Network](https://docs.pypsa.org/latest/examples/scigrid-lopf-then-pf/)*

In [1]:
import datetime as dt
from typing import cast

import numpy as np
import pandas as pd
import pypsa

import typsa
from typsa.components.generator import Generator
from typsa.components.line import Line
from typsa.components.storage_unit import StorageUnit
from typsa.time_variation import Series, TimestampSnapshots

In [2]:
pypsa_network = pypsa.examples.scigrid_de()

contingency_factor = 0.7
pypsa_network.lines.s_max_pu = contingency_factor
pypsa_network.lines.loc[["316", "527", "602"], "s_nom"] = 1715

pypsa_network.generators["control"] = "PV"
pypsa_network.generators.loc[pypsa_network.generators["bus"] == "492", "control"] = "PQ"

network = typsa.Network.from_pypsa_network(pypsa_network, TimestampSnapshots)

INFO:pypsa.network.io:Retrieving network data from https://github.com/PyPSA/PyPSA/raw/v1.0.7/examples/networks/scigrid-de/scigrid-de.nc.
INFO:pypsa.network.io:New version 1.1.2 available! (Current: 1.0.7)
INFO:pypsa.network.io:Imported network 'SciGrid-DE' has buses, carriers, generators, lines, loads, storage_units, transformers


In [3]:
generation_capacity_by_carrier: dict[str | None, float] = {}
for g in network.generators.values():
    if isinstance(g, Generator) and g.p_nom is not None:
        generation_capacity_by_carrier[g.carrier] = (
            generation_capacity_by_carrier.get(g.carrier, 0) + g.p_nom
        )
print(pd.Series(generation_capacity_by_carrier).round(1).to_frame("p_nom"))

                 p_nom
Gas            23913.1
Hard Coal      25312.6
Run of River    3999.1
Waste           1645.9
Brown Coal     20879.5
Oil             2710.2
Storage Hydro   1445.0
Other           3027.8
Multiple         152.7
Nuclear        12068.0
Geothermal        31.7
Wind Offshore   2973.5
Wind Onshore   37339.9
Solar          37041.5


In [4]:
storage_capacity_by_carrier: dict[str | None, float] = {}
for su in network.storage_units.values():
    if isinstance(su, StorageUnit) and su.p_nom is not None:
        storage_capacity_by_carrier[su.carrier] = (
            storage_capacity_by_carrier.get(su.carrier, 0) + su.p_nom
        )
print(pd.Series(storage_capacity_by_carrier).round(1).to_frame("p_nom"))

               p_nom
Pumped Hydro  9179.5


In [5]:
optimized_network = network.optimize_with_rolling_horizon(horizon=4, overlap=0)

INFO:pypsa.optimization.abstract:Optimizing network for snapshot horizon [2011-01-01 00:00:00:2011-01-01 03:00:00] (1/6).
Index(['2', '5', '10', '12', '13', '15', '18', '20', '22', '24', '26', '30',
       '32', '37', '42', '46', '52', '56', '61', '68', '69', '74', '78', '86',
       '87', '94', '95', '96', '99', '100', '104', '105', '106', '107', '117',
       '120', '123', '124', '125', '128', '129', '138', '143', '156', '157',
       '159', '160', '165', '184', '191', '195', '201', '220', '231', '232',
       '233', '236', '247', '248', '250', '251', '252', '261', '263', '264',
       '267', '272', '279', '281', '282', '292', '303', '307', '308', '312',
       '315', '317', '322', '332', '334', '336', '338', '351', '353', '360',
       '362', '382', '384', '385', '391', '403', '404', '413', '421', '450',
       '458'],
      dtype='str', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.06s
INFO:linopy.constants: Optimization successful

Running HiGHS 1.12.0 (git hash: 755a8e0): Copyright (c) 2025 HiGHS under MIT licence terms
LP linopy-problem-q1dgdjy8 has 23828 rows; 9940 cols; 43906 nonzeros
Coefficient ranges:
  Matrix  [1e-02, 2e+02]
  Cost    [3e+00, 1e+02]
  Bound   [0e+00, 0e+00]
  RHS     [7e-08, 6e+03]
Presolving model
3039 rows, 6674 cols, 18332 nonzeros  0s
2307 rows, 5926 cols, 17304 nonzeros  0s
Dependent equations search running on 2078 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
2078 rows, 5102 cols, 16392 nonzeros  0s
Presolve reductions: rows 2078(-21750); columns 5102(-4838); nonzeros 16392(-27514) 
Solving the presolved LP
Using EKK dual simplex solver - serial
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Ph1: 0(0) 0s
       2700     1.4496872502e+06 Pr: 0(0); Du: 0(4.64517e-13) 0s
       2700     1.4496872502e+06 Pr: 0(0); Du: 0(4.64517e-13) 0s

Performed postsolve
Solving th

INFO:pypsa.optimization.abstract:Optimizing network for snapshot horizon [2011-01-01 04:00:00:2011-01-01 07:00:00] (2/6).
Index(['2', '5', '10', '12', '13', '15', '18', '20', '22', '24', '26', '30',
       '32', '37', '42', '46', '52', '56', '61', '68', '69', '74', '78', '86',
       '87', '94', '95', '96', '99', '100', '104', '105', '106', '107', '117',
       '120', '123', '124', '125', '128', '129', '138', '143', '156', '157',
       '159', '160', '165', '184', '191', '195', '201', '220', '231', '232',
       '233', '236', '247', '248', '250', '251', '252', '261', '263', '264',
       '267', '272', '279', '281', '282', '292', '303', '307', '308', '312',
       '315', '317', '322', '332', '334', '336', '338', '351', '353', '360',
       '362', '382', '384', '385', '391', '403', '404', '413', '421', '450',
       '458'],
      dtype='str', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.06s
INFO:linopy.constants: Optimization successful

Running HiGHS 1.12.0 (git hash: 755a8e0): Copyright (c) 2025 HiGHS under MIT licence terms
LP linopy-problem-2ba6gjzh has 23828 rows; 9940 cols; 43906 nonzeros
Coefficient ranges:
  Matrix  [1e-02, 2e+02]
  Cost    [3e+00, 1e+02]
  Bound   [0e+00, 0e+00]
  RHS     [2e-11, 6e+03]
Presolving model
3072 rows, 6884 cols, 18635 nonzeros  0s
2162 rows, 5946 cols, 17429 nonzeros  0s
Dependent equations search running on 2095 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
2095 rows, 5226 cols, 16614 nonzeros  0s
Presolve reductions: rows 2095(-21733); columns 5226(-4714); nonzeros 16614(-27292) 
Solving the presolved LP
Using EKK dual simplex solver - serial
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Ph1: 0(0) 0s
       2813     8.7376390233e+05 Pr: 0(0); Du: 0(8.88178e-16) 0s

Performed postsolve
Solving the original LP from the solution after postsolve

Model name      

INFO:pypsa.optimization.abstract:Optimizing network for snapshot horizon [2011-01-01 08:00:00:2011-01-01 11:00:00] (3/6).
Index(['2', '5', '10', '12', '13', '15', '18', '20', '22', '24', '26', '30',
       '32', '37', '42', '46', '52', '56', '61', '68', '69', '74', '78', '86',
       '87', '94', '95', '96', '99', '100', '104', '105', '106', '107', '117',
       '120', '123', '124', '125', '128', '129', '138', '143', '156', '157',
       '159', '160', '165', '184', '191', '195', '201', '220', '231', '232',
       '233', '236', '247', '248', '250', '251', '252', '261', '263', '264',
       '267', '272', '279', '281', '282', '292', '303', '307', '308', '312',
       '315', '317', '322', '332', '334', '336', '338', '351', '353', '360',
       '362', '382', '384', '385', '391', '403', '404', '413', '421', '450',
       '458'],
      dtype='str', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.06s
INFO:linopy.constants: Optimization successful

...

Running HiGHS 1.12.0 (git hash: 755a8e0): Copyright (c) 2025 HiGHS under MIT licence terms
LP linopy-problem-8r9yh_vs has 23828 rows; 9940 cols; 43906 nonzeros
Coefficient ranges:
  Matrix  [1e-02, 2e+02]
  Cost    [3e+00, 1e+02]
  Bound   [0e+00, 0e+00]
  RHS     [4e-10, 6e+03]
Presolving model
3220 rows, 8585 cols, 20504 nonzeros  0s
2230 rows, 7561 cols, 19042 nonzeros  0s
2151 rows, 5406 cols, 16640 nonzeros  0s
Dependent equations search running on 2124 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
2124 rows, 5361 cols, 16684 nonzeros  0s
Presolve reductions: rows 2124(-21704); columns 5361(-4579); nonzeros 16684(-27222) 
Solving the presolved LP
Using EKK dual simplex solver - serial
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Ph1: 0(0) 0s
       2740     1.4552326783e+06 Pr: 0(0); Du: 0(2.66454e-15) 0s

Performed postsolve
Solving the original LP from the s

INFO:pypsa.optimization.abstract:Optimizing network for snapshot horizon [2011-01-01 16:00:00:2011-01-01 19:00:00] (5/6).
Index(['2', '5', '10', '12', '13', '15', '18', '20', '22', '24', '26', '30',
       '32', '37', '42', '46', '52', '56', '61', '68', '69', '74', '78', '86',
       '87', '94', '95', '96', '99', '100', '104', '105', '106', '107', '117',
       '120', '123', '124', '125', '128', '129', '138', '143', '156', '157',
       '159', '160', '165', '184', '191', '195', '201', '220', '231', '232',
       '233', '236', '247', '248', '250', '251', '252', '261', '263', '264',
       '267', '272', '279', '281', '282', '292', '303', '307', '308', '312',
       '315', '317', '322', '332', '334', '336', '338', '351', '353', '360',
       '362', '382', '384', '385', '391', '403', '404', '413', '421', '450',
       '458'],
      dtype='str', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.06s
INFO:linopy.constants: Optimization successful

Running HiGHS 1.12.0 (git hash: 755a8e0): Copyright (c) 2025 HiGHS under MIT licence terms
LP linopy-problem-ipwkwupv has 23828 rows; 9940 cols; 43906 nonzeros
Coefficient ranges:
  Matrix  [1e-02, 2e+02]
  Cost    [3e+00, 1e+02]
  Bound   [0e+00, 0e+00]
  RHS     [7e-10, 6e+03]
Presolving model
3089 rows, 6987 cols, 18754 nonzeros  0s
2147 rows, 6007 cols, 17627 nonzeros  0s
Dependent equations search running on 2079 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
2079 rows, 5245 cols, 16750 nonzeros  0s
Presolve reductions: rows 2079(-21749); columns 5245(-4695); nonzeros 16750(-27156) 
Solving the presolved LP
Using EKK dual simplex solver - serial
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Ph1: 0(0) 0s
       2842     2.6469204952e+06 Pr: 0(0); Du: 0(8.88178e-14) 0s

Performed postsolve
Solving the original LP from the solution after postsolve

Model name      

INFO:pypsa.optimization.abstract:Optimizing network for snapshot horizon [2011-01-01 20:00:00:2011-01-01 23:00:00] (6/6).
Index(['2', '5', '10', '12', '13', '15', '18', '20', '22', '24', '26', '30',
       '32', '37', '42', '46', '52', '56', '61', '68', '69', '74', '78', '86',
       '87', '94', '95', '96', '99', '100', '104', '105', '106', '107', '117',
       '120', '123', '124', '125', '128', '129', '138', '143', '156', '157',
       '159', '160', '165', '184', '191', '195', '201', '220', '231', '232',
       '233', '236', '247', '248', '250', '251', '252', '261', '263', '264',
       '267', '272', '279', '281', '282', '292', '303', '307', '308', '312',
       '315', '317', '322', '332', '334', '336', '338', '351', '353', '360',
       '362', '382', '384', '385', '391', '403', '404', '413', '421', '450',
       '458'],
      dtype='str', name='name')
INFO:linopy.model: Solve problem using Highs solver
INFO:linopy.io: Writing time: 0.06s
INFO:linopy.constants: Optimization successful

Running HiGHS 1.12.0 (git hash: 755a8e0): Copyright (c) 2025 HiGHS under MIT licence terms
LP linopy-problem-98m6g7w8 has 23828 rows; 9940 cols; 43906 nonzeros
Coefficient ranges:
  Matrix  [1e-02, 2e+02]
  Cost    [3e+00, 1e+02]
  Bound   [0e+00, 0e+00]
  RHS     [9e-11, 6e+03]
Presolving model
3085 rows, 6973 cols, 18738 nonzeros  0s
2156 rows, 6006 cols, 17588 nonzeros  0s
Dependent equations search running on 2085 equations with time limit of 1000.00s
Dependent equations search removed 0 rows and 0 nonzeros in 0.00s (limit = 1000.00s)
2085 rows, 5260 cols, 16738 nonzeros  0s
Presolve reductions: rows 2085(-21743); columns 5260(-4680); nonzeros 16738(-27168) 
Solving the presolved LP
Using EKK dual simplex solver - serial
  Iteration        Objective     Infeasibilities num(sum)
          0     0.0000000000e+00 Ph1: 0(0) 0s
       2805     2.1429568887e+06 Pr: 0(0); Du: 0(3.55271e-15) 0s

Performed postsolve
Solving the original LP from the solution after postsolve

Model name      

In [6]:
print(
    optimized_network.dynamic_results.of_all_generators.p.T.groupby(
        {g.name: g.carrier for g in optimized_network.generators.values()}
    )
    .sum()
    .T
)

name                   Brown Coal          Gas  Geothermal     Hard Coal  \
snapshot                                                                   
2011-01-01 00:00:00   8954.695746   596.277945         0.0   9508.907684   
2011-01-01 01:00:00   7740.822350   102.731042         0.0   8543.988561   
2011-01-01 02:00:00   7453.596973    87.387948         0.0   6806.982171   
2011-01-01 03:00:00   6969.683154    81.111447         0.0   5886.411673   
2011-01-01 04:00:00   6438.037241    71.564784         0.0   4677.000222   
2011-01-01 05:00:00   5928.410387    53.962358         0.0   2609.944440   
2011-01-01 06:00:00   5842.342126    54.091201         0.0   2642.906648   
2011-01-01 07:00:00   5532.020781    57.889006         0.0   2727.486778   
2011-01-01 08:00:00   6053.121099     0.000000         0.0   1453.973538   
2011-01-01 09:00:00   6101.341441     0.000000         0.0   1277.878373   
2011-01-01 10:00:00   7267.499030     0.000000         0.0   2469.571971   
2011-01-01 1

In [7]:
optimized_network.dynamic_results.of_all_storage_units.p.sum(axis="columns")

snapshot
2011-01-01 00:00:00    -219.100000
2011-01-01 01:00:00    -219.100000
2011-01-01 02:00:00    -219.100000
2011-01-01 03:00:00    -821.000000
2011-01-01 04:00:00    -102.450542
2011-01-01 05:00:00    -119.100000
2011-01-01 06:00:00    -185.838932
2011-01-01 07:00:00    -219.100000
2011-01-01 08:00:00     -76.520040
2011-01-01 09:00:00     -98.800000
2011-01-01 10:00:00       0.000000
2011-01-01 11:00:00     -27.348162
2011-01-01 12:00:00    -729.865025
2011-01-01 13:00:00    -285.416620
2011-01-01 14:00:00     184.678994
2011-01-01 15:00:00     988.591670
2011-01-01 16:00:00    -538.039566
2011-01-01 17:00:00     166.282538
2011-01-01 18:00:00    1186.649039
2011-01-01 19:00:00     824.760632
2011-01-01 20:00:00     -74.167226
2011-01-01 21:00:00     -36.636098
2011-01-01 22:00:00     170.000000
2011-01-01 23:00:00       0.000000
dtype: float64

In [8]:
optimized_network.dynamic_results.of_all_storage_units.state_of_charge.sum(
    axis="columns"
)

snapshot
2011-01-01 00:00:00     208.145000
2011-01-01 01:00:00     416.290000
2011-01-01 02:00:00     624.435000
2011-01-01 03:00:00    1404.385000
2011-01-01 04:00:00    1501.713015
2011-01-01 05:00:00    1614.858015
2011-01-01 06:00:00    1791.405000
2011-01-01 07:00:00    1999.550000
2011-01-01 08:00:00    2072.244038
2011-01-01 09:00:00    2166.104038
2011-01-01 10:00:00    2166.104038
2011-01-01 11:00:00    2175.845773
2011-01-01 12:00:00    2869.217547
2011-01-01 13:00:00    3140.363337
2011-01-01 14:00:00    2937.628554
2011-01-01 15:00:00    1897.005743
2011-01-01 16:00:00    2389.323777
2011-01-01 17:00:00    2190.957549
2011-01-01 18:00:00     941.853297
2011-01-01 19:00:00      73.684211
2011-01-01 20:00:00     144.143075
2011-01-01 21:00:00     178.947368
2011-01-01 22:00:00       0.000000
2011-01-01 23:00:00       0.000000
dtype: float64

In [9]:
now = dt.datetime.fromisoformat("2011-01-01 04:00:00")
loading = optimized_network.dynamic_results.of_all_lines.p0.loc[now] / pd.Series(
    {x.name: x.s_nom for x in optimized_network.lines.values() if isinstance(x, Line)}
)
print(loading.describe())

count    852.000000
mean      -0.003145
std        0.260237
min       -0.700000
25%       -0.128281
50%        0.003209
75%        0.121985
max        0.700000
dtype: float64


In [10]:
print(optimized_network.dynamic_results.of_all_buses.marginal_price.loc[now].describe())

count    585.000000
mean      15.737598
std       10.941995
min      -10.397824
25%        6.992120
50%       15.841190
75%       25.048186
max       52.150120
Name: 2011-01-01 04:00:00, dtype: float64


In [11]:
carrier = "Wind Onshore"
capacity = generation_capacity_by_carrier[carrier]
p_available = {
    g.name: g.p_max_pu.to_pandas() * g.p_nom
    for g in network.generators.values()
    if isinstance(g, Generator)
    and isinstance(g.p_max_pu, Series)
    and g.p_nom is not None
}
generators = network.generators
p_available_by_carrier = {
    c: sum(p_available.get(g.name, 0) for g in generators.values() if g.carrier == c)
    for c in network.carriers.keys()
}
p_by_carrier = (
    optimized_network.dynamic_results.of_all_generators.p.T.groupby(
        {g.name: g.carrier for g in network.generators.values()}
    )
    .sum()
    .T
)
p_curtailed_by_carrier = pd.DataFrame(p_available_by_carrier) - p_by_carrier
p_df = pd.DataFrame(
    {
        f"{carrier} capacity": capacity,
        f"{carrier} available": p_available_by_carrier[carrier],
        f"{carrier} dispatched": p_by_carrier[carrier],
        f"{carrier} curtailed": p_curtailed_by_carrier[carrier],
    }
)

In [12]:
pf_results, pf_info = optimized_network.pf()

INFO:pypsa.network.power_flow:Performing non-linear load-flow on AC sub-network <pypsa.networks.SubNetwork object at 0x12f40e690> for snapshots DatetimeIndex(['2011-01-01 00:00:00', '2011-01-01 01:00:00',
               '2011-01-01 02:00:00', '2011-01-01 03:00:00',
               '2011-01-01 04:00:00', '2011-01-01 05:00:00',
               '2011-01-01 06:00:00', '2011-01-01 07:00:00',
               '2011-01-01 08:00:00', '2011-01-01 09:00:00',
               '2011-01-01 10:00:00', '2011-01-01 11:00:00',
               '2011-01-01 12:00:00', '2011-01-01 13:00:00',
               '2011-01-01 14:00:00', '2011-01-01 15:00:00',
               '2011-01-01 16:00:00', '2011-01-01 17:00:00',
               '2011-01-01 18:00:00', '2011-01-01 19:00:00',
               '2011-01-01 20:00:00', '2011-01-01 21:00:00',
               '2011-01-01 22:00:00', '2011-01-01 23:00:00'],
              dtype='datetime64[ns]', name='snapshot', freq=None)


In [13]:
print((~pf_info.converged).any().any())

False


In [14]:
print(
    (
        pf_results.of_all_lines.p0.loc[now]
        / pd.Series(
            {x.name: x.s_nom for x in network.lines.values() if isinstance(x, Line)}
        )
    ).describe()
)

count    852.000000
mean       0.000226
std        0.262545
min       -0.813767
25%       -0.123980
50%        0.003103
75%        0.123851
max        0.827600
dtype: float64


In [15]:
print((pypsa_network.lines_t.p0.loc[now] / pypsa_network.lines.s_nom).describe())

count    0.0
mean     NaN
std      NaN
min      NaN
25%      NaN
50%      NaN
75%      NaN
max      NaN
dtype: float64


In [16]:
pd.Series(
    {
        line.name: np.rad2deg(
            cast(float, pf_results.of_all_buses.v_ang.at[now, line.bus0])
            - cast(float, pf_results.of_all_buses.v_ang.at[now, line.bus1])
        )
        for line in network.lines.values()
    }
).describe()

count    852.000000
mean      -0.022249
std        2.386780
min      -12.158079
25%       -0.465772
50%        0.001604
75%        0.534861
max       17.959258
dtype: float64

In [17]:
print(cast(pd.Series, pf_results.of_all_buses.q.loc[now]).to_frame())

           2011-01-01 04:00:00
name                          
1                  -122.360489
2                     0.000000
3                   -43.669934
4                  -118.776425
5                     0.000000
...                        ...
404_220kV           -13.676996
413_220kV            -2.377202
421_220kV           -12.830586
450_220kV            -9.713586
458_220kV           -12.147480

[585 rows x 1 columns]
